In [1]:
import pandas as pd
import numpy as np
from itertools import  product
import matplotlib.pyplot as plt

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

### Final Predictions - Significant Features

In [1]:
import pandas as pd
import numpy as np
from itertools import  product
import matplotlib.pyplot as plt

In [2]:
df_d_oot = pd.read_excel(r"D:\Temp\Q3_FY_26_As_Input_Data.xlsx")

In [3]:
df_d_oot.shape

(1043, 12)

In [4]:
df_d_oot.reset_index(inplace=True)

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Flatten, Concatenate, Reshape, RepeatVector
from tensorflow.keras.models import Model
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
import tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix,roc_auc_score
from tensorflow.keras.metrics import AUC, Precision
import os
from scipy.stats import skew

In [6]:
best_transforms = {'TMR': 'yeo-johnson',
 'Momentum': 'yeo-johnson',
 'Beta': 'yeo-johnson',
 'Alpha': 'yeo-johnson',
 'EP': 'boxcox',
 'BP': 'yeo-johnson',
 'SP': 'boxcox',
 'Dividend_Flag': 'original',
 'DE': 'yeo-johnson',
 'EPS': 'yeo-johnson',
 'Zscore': 'yeo-johnson'}

In [7]:
df_d_oot.replace([np.inf, -np.inf], 0, inplace=True)

In [8]:
df_d_oot['EP'] = df_d_oot['EP'].fillna(0)

In [9]:
df_d_oot['SP'].max()

np.float64(78.031596135344)

In [10]:
df_d_oot['SP'] = df_d_oot['SP'].fillna(0)

In [11]:
# list of params being used in modelling
sign_feat = ['TMR', 'Momentum','Beta','Alpha','Zscore','EP', 'BP', 'SP', 'Dividend_Flag', 'DE', 'EPS']

In [12]:
df_d_oot.columns

Index(['index', 'Ticker', 'TMR', 'Momentum', 'Beta', 'Alpha', 'Zscore', 'EP',
       'BP', 'SP', 'Dividend_Flag', 'DE', 'EPS'],
      dtype='object')

In [13]:
import pickle 
filename = r'D:\Temp\top200_xform_lag\bi_lstm_top200_xform_standard_scaler_16qtr.pkl'
label_encoder = pickle.load(open(filename,'rb'))

In [14]:
label_encoder.get_params()

{'copy': True, 'with_mean': True, 'with_std': True}

In [15]:
df_d_oot

,index,Ticker,TMR,Momentum,Beta,Alpha,Zscore,EP,BP,SP,Dividend_Flag,DE,EPS
0,0,reliance,NaN,-173.60,2.485380,-0.008128,2.109184,0.039105,0.412825,0.470369,1.0,35.650840,61.410000
1,1,tcs,-24.283356,-500.10,-2.701944,-0.304874,1.062028,0.043274,0.093567,0.222139,1.0,9.975636,136.635805
2,2,hdfcb,NaN,-119.85,0.436787,-0.114305,0.453581,0.047448,0.356783,0.317445,1.0,106.230822,47.260000
3,3,bharti,19.685390,-222.30,0.500251,-0.098100,1.533825,0.024543,0.091796,0.162065,1.0,129.719059,65.964178
4,4,icicibc,NaN,-9.90,0.130425,-0.017140,1.081623,0.055045,0.349728,0.317722,1.0,82.082240,75.100000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,1038,yasho,-10.554808,-44.00,8.599962,0.367665,0.023882,0.003699,0.244927,0.387576,1.0,138.804780,5.320000
1039,1039,gocl,-8.696407,-42.10,-8.789144,-0.574609,-0.612221,0.000000,1.081261,0.380435,1.0,70.750344,31.710000
1040,1040,mbap,NaN,NaN,NaN,NaN,NaN,0.015549,0.109237,0.286495,1.0,78.833403,6.560000
1041,1041,jind,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.000000,0.0,69.159222,18.670000


In [16]:
# to remove column named index - which is intermittent - ignore if there is no such column
df_d_oot.drop(columns=['index'],inplace=True)

In [17]:
label_encoder = LabelEncoder()
label_encoder.classes_ = np.load(r"D:\Temp\top200_xform_lag\bi_lstm_top200_xform_label_encoder_16qtr.npy",allow_pickle=True)

In [18]:
len(label_encoder.classes_)

942

In [20]:
#  filtering out unknown labels - that were not training by encoder

In [19]:
df_d_oot2 = df_d_oot[np.where(df_d_oot['Ticker'].isin(label_encoder.classes_),True,False)]

In [20]:
# ...existing code...
from sklearn.preprocessing import PowerTransformer
from scipy.stats import boxcox
import numpy as np


def transform_feature(col, x, method):
    if method == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson')
        return pt.fit_transform(x.values.reshape(-1, 1)).flatten()
    elif method == 'boxcox':
        # Box-Cox requires strictly positive values
        # Add a small constant if min(x) == 0
        x_pos = x.copy()
        if (x_pos <= 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return boxcox(x_pos)[0]
    elif method == 'log1p':
        x_pos = x.copy()
        if (x_pos < 0).any():
            x_pos = x_pos - x_pos.min() + 1e-6
        return np.log1p(x_pos)
    else:
        return x.values

# 2. Apply transforms to train and test
df_d_oot_trans = df_d_oot2.copy()
# df_test_trans = df_test.copy()

for col in sign_feat:
    method = best_transforms.get(col, 'none')
    if method != 'none':
        df_d_oot_trans[col] = transform_feature(col, df_d_oot2[col], method)
        # df_test_trans[col] = transform_feature(col, df_test[col], method)

# 3. Scaling
# scaler = StandardScaler()
import pickle 
filename = r'D:\Temp\top200_xform_lag\bi_lstm_top200_xform_standard_scaler_16qtr.pkl'
scaler = pickle.load(open(filename,'rb'))

train_data = scaler.fit_transform(df_d_oot_trans[sign_feat])
# test_data = scaler.transform(df_test_trans[sign_feat])

df_d_oot1 = pd.DataFrame(train_data, columns=sign_feat)
# df_test1 = pd.DataFrame(test_data, columns=sign_feat)

# for col in ['Ticker', 'Target', 'Quarter']:
#     df_d_oot1[col] = df_d_oot[col]
    # df_test1[col] = df_test[col]

# 4. Prepare X_train, y_train, X_test, y_test
features = sign_feat.copy()
X_train = df_d_oot1[features].values
# y_train = df_d_oot1['Target'].values
# X_test = df_test1[features].values
# y_test = df_test1['Target'].values

# If your model expects 3D input (samples, timesteps, features):
# X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
# X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("X_train shape:", X_train.shape)
# print("y_train shape:", y_train.shape)
# print("X_test shape:", X_test.shape)
# print("y_test shape:", y_test.shape)
# ...existing code...

X_train shape: (942, 11)


In [21]:
df_d_oot2.reset_index(drop=False,inplace=True)

In [22]:
df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])

C:\Users\43737\AppData\Local\Temp\ipykernel_37868\137447066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Ticker_Encoded'] = label_encoder.transform(df_d_oot2['Ticker'])


In [23]:
def create_ticker_embedding(input_dim, output_dim):
    return Embedding(input_dim=input_dim, output_dim=output_dim, input_length=1)

In [24]:
# Now, the LabelEncoder is aware of all the possible tickers in all datasets
embedding_size = len(label_encoder.classes_)  # Size of the embedding

# You can now proceed with creating your ticker embedding layer and other steps
ticker_embedding_layer = create_ticker_embedding(input_dim=embedding_size, output_dim=10)


d:\Python\Final Quant Fund\tf_219_env\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
ticker_encoded_oot = df_d_oot2['Ticker_Encoded'].values

In [26]:

def calculate_metrics(y_true, y_pred_label,y_pred_prob):
    accuracy = accuracy_score(y_true, y_pred_label)
    precision = precision_score(y_true, y_pred_label)
    recall = recall_score(y_true, y_pred_label)
    f1 = f1_score(y_true, y_pred_label)
    confusion = confusion_matrix(y_true,y_pred_label)
    roc_score = roc_auc_score(y_true,y_pred_prob)
    
    return {
        'accuracy': accuracy,
        'roc':roc_score,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix':f'{confusion}'
    }


def process_and_evaluate1(model, X_train, y_train,X_test, y_test, ticker_encoded_train,ticker_encoded_test,epochs,checkpoint_path,batch_size,k=5):
    
    
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    for train_index, val_index in kf.split(X_train):
        X_train1, X_val = X_train[train_index], X_train[val_index]
        y_train1, y_val = y_train[train_index], y_train[val_index]
        ticker_train, ticker_val = ticker_encoded_train[train_index], ticker_encoded_train[val_index]

    early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
    cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
                                                    save_weights_only=False,save_freq=epochs*batch_size,
                                                    verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.25, patience=7, min_lr=1e-6, verbose=1)
    history = model.fit([ticker_train, X_train1], y_train1,
            validation_data=([ticker_val, X_val], y_val),
            epochs=epochs, batch_size=batch_size,
            callbacks=[early_stop, reduce_lr, cp_callback], verbose=0)

    # history = model.fit([ticker_encoded_train, X_train], y_train, epochs=epochs, batch_size=batch_size, validation_data=([ticker_encoded_test, X_test], y_test),callbacks=[cp_callback,early_stop])
    # model.load_weights(checkpoint_path)

    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='val')
    plt.legend()
    

### Bi-LSTM

In [27]:
from tensorflow.keras.layers import Bidirectional

def build_bi_lstm_top200_xform_model(input_shape, embedding_size, num_tickers):
    ticker_input = Input(shape=(1,), name='ticker_input')
    ticker_embedding = Embedding(input_dim=num_tickers, output_dim=embedding_size, input_length=1)(ticker_input)
    ticker_embedding_flat = Flatten()(ticker_embedding)
    
    continuous_input = Input(shape=(input_shape[1],), name='continuous_input')
    merged = Concatenate()([ticker_embedding_flat, continuous_input])
    reshaped_input = tf.keras.layers.Reshape((1, -1))(merged)
    bi_lstm_layer = Bidirectional(LSTM(16, activation='relu',dropout=0.2))(reshaped_input)
    output = Dense(1, activation='sigmoid')(bi_lstm_layer)    
    model = Model(inputs=[ticker_input, continuous_input], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [28]:
num_tickers = len(label_encoder.classes_)

In [29]:

# Define bi_lstm model
bi_lstm_top200_xform_model = build_bi_lstm_top200_xform_model(X_train.shape, embedding_size, num_tickers)
bi_lstm_top200_xform_model.compile(optimizer='adam', 
                   loss='binary_crossentropy',  # Binary classification problem
                   metrics=[AUC(),Precision()])

In [30]:
checkpoints_path = r"D:\Temp\top200_xform_lag"

In [31]:
bi_lstm_top200_xform_model_loaded = tf.keras.models.load_model(rf'{checkpoints_path}\bi_lstm_top200_xform_16qtr.keras')

In [32]:
pred_probs_oot = bi_lstm_top200_xform_model_loaded.predict([ticker_encoded_oot, X_train])

30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step


In [33]:
np.isnan(pred_probs_oot).sum()

np.int64(245)

In [34]:
preds_oot = pred_probs_oot.reshape(-1)
preds_oot = preds_oot>=0.5
preds_oot = pd.Series(preds_oot)
preds_oot = preds_oot.astype(int)

In [35]:
np.sum(preds_oot)

np.int64(54)

In [36]:
df_d_oot2['Pred_Tag'] = preds_oot
df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)

C:\Users\43737\AppData\Local\Temp\ipykernel_37868\2128540863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Tag'] = preds_oot
C:\Users\43737\AppData\Local\Temp\ipykernel_37868\2128540863.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'] = pred_probs_oot.reshape(-1)


In [37]:
df_d_oot2['Pred_Prob'].fillna(0,inplace=True)

C:\Users\43737\AppData\Local\Temp\ipykernel_37868\2870514544.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)
C:\Users\43737\AppData\Local\Temp\ipykernel_37868\2870514544.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_d_oot2['Pred_Prob'].fillna(0,inplace=True)


In [38]:
df_d_oot2.head()

,index,Ticker,TMR,Momentum,Beta,Alpha,Zscore,EP,BP,SP,Dividend_Flag,DE,EPS,Ticker_Encoded,Pred_Tag,Pred_Prob
0,0,reliance,NaN,-173.60,2.485380,-0.008128,2.109184,0.039105,0.412825,0.470369,1.0,35.650840,61.410000,698,0,0.000000
1,1,tcs,-24.283356,-500.10,-2.701944,-0.304874,1.062028,0.043274,0.093567,0.222139,1.0,9.975636,136.635805,834,0,0.277725
2,2,hdfcb,NaN,-119.85,0.436787,-0.114305,0.453581,0.047448,0.356783,0.317445,1.0,106.230822,47.260000,340,0,0.000000
3,3,bharti,19.685390,-222.30,0.500251,-0.098100,1.533825,0.024543,0.091796,0.162065,1.0,129.719059,65.964178,99,0,0.365052
4,4,icicibc,NaN,-9.90,0.130425,-0.017140,1.081623,0.055045,0.349728,0.317722,1.0,82.082240,75.100000,371,0,0.000000


In [39]:
df_d_oot2.to_excel(r"D:\Temp\top200_xform_lag\Preds_as_of_end_of_Q3FY26.xlsx",index=False)